# Imports

In [15]:
import sys
import os
from datetime import datetime

# To define constants
from typing_extensions import Literal

# Langchain LLM model integration
from langchain.chat_models import init_chat_model

import operator
from typing_extensions import Optional, Annotated, List, Sequence

# Differnt Message types for defining chat model ip and op
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, get_buffer_string

# Langgraph Entities to build and design control flow
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.types import Command

from langgraph.graph.message import add_messages
# For runtime validation on state and output schemas
from pydantic import BaseModel, Field

# Custom Imports
sys.path.append(os.path.abspath("../src"))
from prompts import clarify_with_user_instructions, transform_messages_into_research_topic_prompt

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

True

# Defining State and Schemas

## State Definitions

In [16]:
class AgentInputState(MessagesState):
    """Input state with only messages from user input."""
    pass

class AgentMasterState(MessagesState):
    """
    Main state for the full multi-agent research system.
    Extends MessagesState with additional fields for research coordination.
    """

    # Research brief generated from user conversation history
    research_brief: Optional[str]
    # Messages exchanged with the supervisor agent for coordination
    supervisor_messages: Annotated[Sequence[BaseMessage], add_messages]
    # Raw unprocessed research notes collected during the research phase
    raw_notes: Annotated[list[str], operator.add] = []
    # Processed and structured notes ready for report generation
    notes: Annotated[list[str], operator.add] = []
    # Final formatted research report
    final_report: str

## Structured Output Schemas

In [17]:
class ClarifyWithUserSchema(BaseModel):
    """Schema for user clarification decision and questions."""
    
    need_clarification: bool = Field(
        description="Whether the user needs to be asked a clarifying question.",
    )
    question: str = Field(
        description="A question to ask the user to clarify the report scope",
    )
    verification: str = Field(
        description="Verify message that we will start research after the user has provided the necessary information.",
    )

class ResearchQuestionSchema(BaseModel):
    """Schema for structured research brief generation."""
    
    research_brief: str = Field(
        description="A research question that will be used to guide the research.",
    )

# Scope Research Workflow

## Setting Up LLM

In [18]:
llm = init_chat_model(
    "openai:gpt-4.1",
    temperature=0, # Factual response
    max_tokens=1000, # OpenRouter free token constraints
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1"
)

## Utility Functions

In [19]:
"""User Clarification and Research Brief Generation.

This module implements the scoping phase of the research workflow, where we:
1. Assess if the user's request needs clarification
2. Generate a detailed research brief from the conversation

The workflow uses structured output to make deterministic decisions about
whether sufficient context exists to proceed with research.
"""
# ===== UTILITY FUNCTIONS =====

def get_today_str() -> str:
    """Get current date in a human-readable format."""
    return datetime.now().strftime("%a %b %-d, %Y")

# ===== WORKFLOW NODES =====

def clarify_with_user(state: AgentMasterState) -> Command[Literal["write_research_brief", "__end__"]]:
    """
    Determine if the user's request contains sufficient information to proceed with research.
    """
    # Set up structured output from the LLM
    structured_output_model = llm.with_structured_output(ClarifyWithUserSchema)

    # Invoke the model with clarification instructions and user conversations
    # get_buffer_string => list of messages to a string
    response = structured_output_model.invoke([HumanMessage(content=clarify_with_user_instructions.format(
                                                                                            messages=get_buffer_string(messages=state.get("messages", [])), 
                                                                                            date=get_today_str()))])
    
    # Route based on clarification need
    # If clarification is needed, ask ; else generate research brief
    if response.need_clarification:
        return Command(
            goto=END, 
            update={"messages": [AIMessage(content=response.question)]}
        )
    else:
        return Command(
            goto="write_research_brief", 
            update={"messages": [AIMessage(content=response.verification)]}
        )

def write_research_brief(state: AgentMasterState):
    """
    Transform the conversation history into a comprehensive research brief.
    """
    # Set up structured output from the LLM
    structured_output_model = model.with_structured_output(ResearchQuestionSchema)
    
    # Generate research brief from conversation history)

    response = structured_output_model.invoke([HumanMessage(content=transform_messages_into_research_topic_prompt.format(
                                                                                            messages=get_buffer_string(messages=state.get("messages", [])), 
                                                                                            date=get_today_str()))])
    
    # Update state with generated research brief and pass it to the supervisor
    return {
        "research_brief": response.research_brief,
        "supervisor_messages": [HumanMessage(content=f"{response.research_brief}.")]
    }


## Graph Construction

In [20]:
# Build the scoping workflow
deep_researcher_builder = StateGraph(AgentMasterState, input_schema=AgentInputState)

# Add workflow nodes
deep_researcher_builder.add_node("clarify_with_user", clarify_with_user)
deep_researcher_builder.add_node("write_research_brief", write_research_brief)

# Add workflow edges
deep_researcher_builder.add_edge(START, "clarify_with_user") # END if further claification needed, else goto write_research_brief
deep_researcher_builder.add_edge("write_research_brief", END)

# Compile the workflow
scope_research = deep_researcher_builder.compile()